[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LD-Shell/beamfeat/blob/main/notebooks/02_search_and_scoring.ipynb)

Run the cells in order (in Colab: Runtime → Run all).

In [ ]:
# In Colab, or wherever beamfeat is not installed yet
%pip install -q "beamfeat[all]"

# Search and scoring

This notebook covers how beamfeat explores the space of expressions, and how
the choice of scorer changes what it finds.

The core problem is that expression space is enormous. With 5 input columns, 5
unary operators, and 4 binary operators, exhaustive expansion to depth 3
produces well over a million candidates, and each must be evaluated before it
can be judged. Beam search avoids this by scoring candidates as they are
generated and extending only the most promising.

In [1]:
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from beamfeat import BeamSearch

rng = np.random.default_rng(0)

n = 500
data = {name: rng.uniform(1.0, 6.0, n) for name in "abcd"}
target = (data["a"] * data["b"]) / data["c"] + rng.normal(0, 0.05, n)

## Running a search directly

`BeamSearch` is the layer beneath the estimators. Using it directly is useful
when you want to inspect the search rather than fit a model.

In [2]:
search = BeamSearch(max_depth=2, beam_width=30, random_state=0)
result = search.run(data, target)

print(result.summary())

beam search: 30 features from 3094 proposed / 3046 evaluated in 0.05s, 2 near-duplicate(s) dropped across depths
  depth 0: proposed 4, evaluated 4, kept 4, best 0.5941 (0.00s)
  depth 1: proposed 56, evaluated 52, kept 30, best 0.8411 (0.00s)
  depth 2: proposed 3038, evaluated 2994, kept 30, best 0.9999 (0.05s)
  top features:
      0.9999  ((a / c) * b)
      0.9435  ((b / c) * (a - c))
      0.9430  ((a / c) + sqrt(b))
      0.9403  ((a / c) * (b - c))
      0.9387  ((a * b) / sqrt(c))
      0.9371  ((b / c) + log(a))
      0.9350  ((a / c) * (b / c))
      0.9302  ((a / c) + (b / c))
      0.9201  ((a + d) * (b / c))
      0.9109  ((b + d) * (a / c))


## The trace

Every search records what happened at each depth. This is how you attribute
cost, and how you notice a beam that collapsed or saturated.

In [3]:
print(f"{'depth':>6} {'proposed':>10} {'evaluated':>10} {'kept':>6} {'best':>8} {'seconds':>8}")
for record in result.trace:
    print(
        f"{record.depth:>6} {record.n_proposed:>10} {record.n_evaluated:>10} "
        f"{record.n_kept:>6} {record.best_score:>8.4f} {record.elapsed:>8.3f}"
    )

 depth   proposed  evaluated   kept     best  seconds
     0          4          4      4   0.5941    0.001
     1         56         52     30   0.8411    0.004
     2       3038       2994     30   0.9999    0.048


## How beam width controls cost

Beam width is the parameter that bounds work. Widening it explores more of the
space at a cost that grows faster than the width, since each expression in the
beam is combined with every expression kept so far; narrowing it risks pruning
a parent whose children would have been valuable.

In [4]:
print(f"{'beam':>6} {'proposed':>10} {'kept':>6} {'seconds':>9}  top feature")
for width in (5, 10, 25, 50, 100):
    outcome = BeamSearch(max_depth=2, beam_width=width, random_state=0).run(data, target)
    top = outcome.names[0] if outcome.names else "-"
    print(
        f"{width:>6} {outcome.n_proposed_total:>10} {len(outcome):>6} "
        f"{outcome.elapsed:>9.3f}  {top}"
    )

  beam   proposed   kept   seconds  top feature
     5        219      5     0.005  ((a / c) * b)
    10        509     10     0.009  ((a / c) * b)
    25       2265     25     0.035  ((a / c) * b)
    50       3490     50     0.081  ((a / c) * b)
   100       3490    100     0.062  ((a / c) * b)


Note that cost grows with beam width but the *quality* of the top feature
plateaus quickly. The proposals stop growing at 50 only because this problem
runs out of parents: four columns yield 52 evaluated depth-1 candidates (the
trace above), fewer than 50 of them survive the redundancy check described
below, and a wider beam has nothing more to extend. Timings this short are
noisy, so the proposed column is the steadier measure of cost. In practice a
moderate beam is usually enough, and the remaining budget is better spent on
depth.

## How depth controls expressiveness

Some relationships simply cannot be expressed below a certain depth. `(a * b) /
c` needs depth 2: one step to build `a * b`, another to divide by `c`.

In [5]:
for depth in (0, 1, 2, 3):
    outcome = BeamSearch(max_depth=depth, beam_width=25, random_state=0).run(data, target)
    found = any(all(token in name for token in "abc") for name in outcome.names)
    print(f"depth {depth}: {len(outcome):>3} features, "
          f"three-way interaction found: {found}")

depth 0:   4 features, three-way interaction found: False
depth 1:  25 features, three-way interaction found: False
depth 2:  25 features, three-way interaction found: True
depth 3:  25 features, three-way interaction found: True


## Workers and memory

At the defaults a depth may propose up to 200,000 candidates
(`max_candidates_per_depth`), so the search evaluates and scores a depth in
chunks and keeps only the scores; the candidates the redundancy check reaches,
usually a small share of the round, are evaluated again. Memory stays bounded
however many candidates a depth proposes. `chunk_size` sets how many candidates
make a chunk, by default as many as fit in 8 MiB. Mutual information is the
exception: it draws its noise for the whole batch, so it still scores a depth
at once.

`n_jobs` spreads the chunks over joblib workers: threads, since numpy releases
the GIL, or processes for gradient boosting, whose model fits hold it. `-1`
uses every core. A depth of fewer than 4096 candidates stays in the calling
thread unless the scorer fits models, since starting workers would cost more
than it saves, so on the four-column problem above `n_jobs` changes nothing.
The estimators take the same argument.

Neither setting changes the result, for the scorers that draw random numbers
too: no seed depends on a worker or a chunk. On forty columns, where both
depths are large enough for workers:

In [6]:
# A generator of its own, so that this cell leaves the draws of the cells
# below unchanged.
wide_rng = np.random.default_rng(1)
wide = {f"x{index}": wide_rng.uniform(1.0, 6.0, n) for index in range(40)}
wide_target = wide["x3"] * wide["x17"] / wide["x31"] + wide_rng.normal(0, 0.05, n)

serial = BeamSearch(max_depth=2, beam_width=30, random_state=0).run(wide, wide_target)
print(f"candidates at depth 1: {serial.trace[1].n_proposed}, at depth 2: {serial.trace[2].n_proposed}")
print(f"top feature: {serial.names[0]}")
print()

for name, value in (("n_jobs", 2), ("n_jobs", -1), ("chunk_size", 100)):
    outcome = BeamSearch(max_depth=2, beam_width=30, random_state=0, **{name: value}).run(
        wide, wide_target
    )
    same = outcome.names == serial.names and np.array_equal(outcome.scores, serial.scores)
    label = f"{name}={value}"
    print(f"{label:>14}: same features and scores as n_jobs=1: {same}")

candidates at depth 1: 4880, at depth 2: 7437
top feature: ((x3 / x31) * x17)

      n_jobs=2: same features and scores as n_jobs=1: True
     n_jobs=-1: same features and scores as n_jobs=1: True
chunk_size=100: same features and scores as n_jobs=1: True


Whether workers save time depends on the machine and on the size of the round,
so they are not timed here. The 0.5.0 changelog measures the faster search as a
whole: a default fit on 1000 columns and 600 rows takes 7 s rather than 29 s
and peaks at 0.35 GB of memory rather than 2.6 GB, with the same features,
p-values and predictions.

## Choosing a scorer

Four scoring strategies are available, and they differ in what they can detect
and what they cost.

- `"correlation"` — absolute Pearson correlation against the residual. One
  dot product per candidate. The default.
- `"spearman"` — absolute rank correlation against the same residual, for a
  sort per column more. Sees any monotone relationship and is insensitive to
  outlying values.
- `"mutual_information"` — nearest-neighbour mutual information. Detects
  dependence that correlation misses.
- `"gradient_boosting"` — measured out-of-fold predictive improvement. Scores
  what is actually being optimised.

### Where they agree

On a relationship whose nonlinearity is captured by the expression itself, the
first three find the same thing, and correlation is much the cheapest. Gradient
boosting is left out: it fits models for every candidate, which on a round of
this size takes far longer still.

In [7]:
import time

for name in ("correlation", "spearman", "mutual_information"):
    started = time.perf_counter()
    outcome = BeamSearch(scorer=name, max_depth=2, beam_width=20, random_state=0).run(data, target)
    elapsed = time.perf_counter() - started
    print(f"{name:>20}: {elapsed:>6.2f}s   top: {outcome.names[0]}")

         correlation:   0.02s   top: ((a / c) * b)
            spearman:   0.13s   top: ((a / c) * b)
  mutual_information:   5.33s   top: ((a / c) * b)


### Where they differ

Correlation is blind to symmetric relationships. If the target depends on the
*magnitude* of a feature but not its sign, correlation sees nothing while
mutual information sees the dependence clearly.

In [8]:
from beamfeat.scoring import CorrelationScorer, MutualInformationScorer

symmetric_feature = rng.uniform(-3, 3, n)
symmetric_target = symmetric_feature**2 + rng.normal(0, 0.3, n)
noise = rng.normal(size=n)

candidates = np.column_stack([symmetric_feature, noise])

corr = CorrelationScorer().score_batch(candidates, symmetric_target)
mutual = MutualInformationScorer().score_batch(candidates, symmetric_target)

print(f"{'':>12} {'true feature':>14} {'noise':>10}")
print(f"{'correlation':>12} {corr[0]:>14.4f} {corr[1]:>10.4f}")
print(f"{'mutual info':>12} {mutual[0]:>14.4f} {mutual[1]:>10.4f}")

               true feature      noise
 correlation         0.0149     0.0493
 mutual info         1.7938     0.0000


Correlation cannot distinguish the true feature from noise here. Mutual
information can.

In practice this matters less than it appears, because beamfeat applies the
scorer to *transformed* columns. Once `square(x)` has been constructed, a
linear scorer detects it easily — which is why correlation remains a sensible
default despite this weakness.

In [9]:
squared = symmetric_feature**2  # what the search would hand the scorer
print(f"correlation on the raw column:       {CorrelationScorer().score(symmetric_feature, symmetric_target):.4f}")
print(f"correlation on the squared column:   {CorrelationScorer().score(squared, symmetric_target):.4f}")

correlation on the raw column:       0.0149
correlation on the squared column:   0.9926


## Redundancy control

A beam scored purely on individual merit fills with variants of the same
signal. beamfeat scores each candidate against the residual left by what is
already selected as well as against the target, and ranks it on the larger of
the two, so a refinement of a selected feature survives while novelty is still
rewarded. It also drops a candidate too correlated with one already admitted at
the same depth; the final ranking applies the same check across depths.

In [10]:
from beamfeat import Evaluator

for threshold in (0.5, 0.9, 0.999):
    outcome = BeamSearch(
        max_depth=2, beam_width=30, redundancy_threshold=threshold, random_state=0
    ).run(data, target)

    evaluator = Evaluator(data)
    _, matrix = evaluator.evaluate_many(outcome.nodes)
    correlations = np.abs(np.corrcoef(matrix, rowvar=False))
    np.fill_diagonal(correlations, 0.0)

    print(
        f"threshold {threshold:<6}: {len(outcome):>3} features, "
        f"max pairwise correlation {np.nanmax(correlations):.3f}"
    )

threshold 0.5   :  14 features, max pairwise correlation 0.497
threshold 0.9   :  30 features, max pairwise correlation 0.899
threshold 0.999 :  30 features, max pairwise correlation 0.997


## Auditing what was rejected

Candidates excluded during evaluation are recorded rather than silently
dropped. This matters when a search returns less than expected: the log tells
you whether candidates were numerically invalid, degenerate, copies of a column
already there — `abs` of a column that is never negative is the column itself —
or simply never proposed.

In [11]:
signed_data = dict(data)
signed_data["e"] = rng.normal(0, 2, n)  # negative values break log and sqrt

outcome = BeamSearch(max_depth=1, beam_width=20, random_state=0).run(signed_data, target)

print("rejections by reason:")
for reason, count in outcome.evaluation_log.counts().items():
    print(f"  {reason.value:>15}: {count}")

print("\nexamples:")
for record in list(outcome.evaluation_log)[:5]:
    print(f"  {record}")

rejections by reason:
        duplicate: 4
     domain_error: 2

examples:
  abs(a) [duplicate]: 'abs' leaves its operand unchanged
  abs(b) [duplicate]: 'abs' leaves its operand unchanged
  abs(c) [duplicate]: 'abs' leaves its operand unchanged
  abs(d) [duplicate]: 'abs' leaves its operand unchanged
  log(e) [domain_error]: input outside domain of 'log'
